In [1]:
import pandas as pd
import numpy as np

# Json data cleaning: Scouting Reports 2019 - 2023

In [2]:
# Read in the scouting report data for each year
recruits_2019 = pd.read_json(r'../Data/scouting_reports_2019.json')
recruits_2020 = pd.read_json(r'../Data/scouting_reports_2020.json')
recruits_2021 = pd.read_json(r'../Data/scouting_reports_2021.json')
recruits_2022 = pd.read_json(r'../Data/scouting_reports_2022.json')
recruits_2023 = pd.read_json(r'../Data/scouting_reports_2023.json')

Before doing any individual operations on each dataset, we need to filter out all rows that do not have a scouting report

In [3]:
# Filter out rows where the scputing report is null
recruits_2019 = recruits_2019.dropna(subset=['scouting_report'])
recruits_2020 = recruits_2020.dropna(subset=['scouting_report'])
recruits_2021 = recruits_2021.dropna(subset=['scouting_report'])
recruits_2022 = recruits_2022.dropna(subset=['scouting_report'])
recruits_2023 = recruits_2023.dropna(subset=['scouting_report'])

Next, within the data, the team names are inconsistent across players due to various reasons related to the html formatting. To fix this, we have a function below to standardize these names and idenitfies schools that are not colleges so we can review them further. 

In [4]:
SCHOOL_MAP = {
    # FBS / major programs
    'Oklahoma Sooners':             'Oklahoma',
    'Arizona State Sun Devils':     'Arizona State',
    'Arizona State Sun':     'Arizona State',
    'Auburn Tigers':                'Auburn',
    'North Carolina Tar Heels':     'North Carolina',
    'North Carolina Tar':           'North Carolina',
    'TCU Horned Frogs':             'TCU',
    'South Carolina Gamecocks':     'South Carolina',
    'Wisconsin Badgers':            'Wisconsin',
    'Alabama Crimson Tide':         'Alabama',
    'Washington Huskies':           'Washington',
    'Oregon Ducks':                 'Oregon',
    'Arkansas Razorbacks':          'Arkansas',
    'Boise State Broncos':          'Boise State',
    'Michigan Wolverines':          'Michigan',
    'Texas Longhorns':              'Texas',
    'Ole Miss Rebels':              'Ole Miss',
    'California Golden Bears':      'California',
    'Arizona Wildcats':             'Arizona',
    'Florida Gators':               'Florida',
    'Florida State Seminoles':      'Florida State',
    'Tennessee Volunteers':         'Tennessee',
    'Oklahoma State Cowboys':       'Oklahoma State',
    'Minnesota Golden Gophers':     'Minnesota',
    'Minnesota Golden':             'Minnesota',
    'Cincinnati Bearcats':          'Cincinnati',
    'LSU Tigers':                   'LSU',
    'Notre Dame Fighting Irish':    'Notre Dame',
    'Washington State Cougars':     'Washington State',
    'Mississippi State':            'Mississippi State',
    'Toledo Rockets':               'Toledo',
    'Penn State Nittany':           'Penn State',
    'Penn State Nittany Lions':     'Penn State',
    'Texas A&M Aggies':             'Texas A&M',
    'Maryland Terrapins':           'Maryland',
    'Iowa State Cyclones':          'Iowa State',
    'Michigan State Spartans':      'Michigan State',
    'Pittsburgh Panthers':          'Pittsburgh',
    'Clemson Tigers':               'Clemson',
    'Southern Miss Golden':         'Southern Miss',
    'Southern Miss Golden Eagles':  'Southern Miss',
    'UCF Knights':                  'UCF',
    'Baylor Bears':                 'Baylor',
    'SMU Mustangs':                 'SMU',
    'Purdue Boilermakers':          'Purdue',
    'Missouri Tigers':              'Missouri',
    'Miami (OH) RedHawks':          'Miami (OH)',
    'Wake Forest Demon Deacons':    'Wake Forest',
    'BYU Cougars':                  'BYU',
    'Brigham Young Cougars':        'BYU',
    'USF Bulls':                    'USF',
    'Temple Owls':                  'Temple',
    'Miami Hurricanes':             'Miami',
    'Kent State Golden Flashes':    'Kent State',
    'FIU Golden Panthers':          'FIU',
    'Kentucky Wildcats':            'Kentucky',
    "Louisiana Ragin' Cajuns":      'Louisiana',
    "Louisiana Ragin'":             'Louisiana',
    'Northern Iowa Panthers':       'Northern Iowa',
    'Northern Iowa':                'Northern Iowa',
    'Georgia Tech Yellow':          'Georgia Tech',
    'Utah State Aggies':            'Utah State',
    'Nevada Wolf Pack':             'Nevada',
    'Richmond':                     'Richmond',
    'Kansas State Wildcats':        'Kansas State',
    'Texas Tech Red Raiders':       'Texas Tech',
    'Appalachian State':            'Appalachian State',
    'Boston College Eagles':        'Boston College',
    'Georgia State Panthers':       'Georgia State',
    'Memphis Tigers':               'Memphis',
    'Buffalo Bulls':                'Buffalo',
    'Syracuse Orange':              'Syracuse',
    'Old Dominion Monarchs':        'Old Dominion',
    'East Carolina Pirates':        'East Carolina',
    'Virginia Cavaliers':           'Virginia',
    'New Mexico State Aggies':      'New Mexico State',
    'Virginia Tech Hokies':         'Virginia Tech',
    'UCLA Bruins':                  'UCLA',
    'Hawaii Warriors':              'Hawaii',
    'Massachusetts Minutemen':      'Massachusetts',
    'Princeton':                    'Princeton',
    'Indiana Hoosiers':             'Indiana',
    'Ohio State Buckeyes':          'Ohio State',
    'Marshall Thundering':          'Marshall',
    'Marshall Thundering Herd':     'Marshall',
    'Central Arkansas':             'Central Arkansas',
    'UC Davis':                     'UC Davis',
    'Fresno State Bulldogs':        'Fresno State',
    'Northern Illinois':            'Northern Illinois',
    'UNLV Rebels':                  'UNLV',
    'Louisiana Tech Bulldogs':      'Louisiana Tech',
    'Rutgers Scarlet Knights':      'Rutgers',
    'Louisiana-Monroe':             'Louisiana-Monroe',
    'UAB Blazers':                  'UAB',
    'Western Michigan Broncos':     'Western Michigan',
    'Montana Grizzlies':            'Montana',
    'Eastern Kentucky':             'Eastern Kentucky',
    'Arkansas State Red Wolves':    'Arkansas State',
    'San Diego State Aztecs':       'San Diego State',
    'Utah Utes':                    'Utah',
    'North Texas Mean Green':       'North Texas',
    'Texas Southern':               'Texas Southern',
    'Louisville Cardinals':         'Louisville',
    'Akron Zips':                   'Akron',
    'Georgia Southern':             'Georgia Southern',
    'Western Carolina':             'Western Carolina',
    'Chattanooga':                  'Chattanooga',
    'Troy Trojans':                 'Troy',
    'Central Washington':           'Central Washington',
    'Air Force':                    'Air Force',
    'Mercer':                       'Mercer',
    'UTEP Miners':                  'UTEP',
    'Northern Arizona':             'Northern Arizona',
    'UTSA Roadrunners':             'UTSA',
    'USC Trojans':                  'USC',
    'Texas State Bobcats':          'Texas State',
    'Coastal Carolina':             'Coastal Carolina',
    'Harvard':                      'Harvard',
    'Central Michigan':             'Central Michigan',
    'Grand Valley State':           'Grand Valley State',
    'Houston Cougars':              'Houston',
    'Murray State':                 'Murray State',
    'Ball State Cardinals':         'Ball State',
    'Ball State':                   'Ball State',
    'Liberty Flames':               'Liberty',
    'Nicholls State Colonels':      'Nicholls State',
    'Ohio Bobcats':                 'Ohio',
    'Idaho State':                  'Idaho State',
    'San Jose State Spartans':      'San Jose State',
    'Rhode Island':                 'Rhode Island',
    'Oregon State Beavers':         'Oregon State',
    'Rice Owls':                    'Rice',
    'Eastern Michigan':             'Eastern Michigan',
    'Western Michigan':             'Western Michigan',
    'Simon Fraser':                 'Simon Fraser',
    'Navy':                         'Navy',
    'Army':                         'Army',
    'Eastern Washington':           'Eastern Washington',
    'Eastern Washington Eagles':    'Eastern Washington',
    'Western Oregon':               'Western Oregon',
    'Alabama State':                'Alabama State',
    'Kennesaw State Owls':          'Kennesaw State',
    'Texas-Permian Basin':          'Texas-Permian Basin',
    'Lamar':                        'Lamar',
    'Connecticut Huskies':          'Connecticut',
    'South Alabama Jaguars':        'South Alabama',
    'Montana State-Northern':       'Montana State-Northern',
    'Prairie View A&M':             'Prairie View A&M',
    'Tulsa Golden Hurricane':       'Tulsa',
    'Holy Cross':                   'Holy Cross',
    'Fordham':                      'Fordham',
    'Dartmouth':                    'Dartmouth',
    'Stephen F. Austin':            'Stephen F. Austin',
    'Portland State':               'Portland State',
    'Idaho':                        'Idaho',
    'Georgia Bulldogs':             'Georgia',
    'Georgetown':                   'Georgetown',
    'New Hampshire':                'New Hampshire',
    'Illinois State':               'Illinois State',
    'Austin Peay':                  'Austin Peay',
    'Yale':                         'Yale',
    'East Tennessee State':         'East Tennessee State',
    'Wyoming Cowboys':              'Wyoming',
    'Vanderbilt Commodores':        'Vanderbilt',
    'Duke Blue Devils':             'Duke',
    'Georgia Tech Yellow Jackets':  'Georgia Tech',
    'Colorado Buffaloes':           'Colorado',
    'Northwestern':                 'Northwestern',
    'Northwestern Wildcats':        'Northwestern',
    'West Virginia Mountaineers':   'West Virginia',
    'West Virginia':                'West Virginia',
    'Mississippi State Bulldogs':   'Mississippi State',
    'North Dakota State Bison':     'North Dakota State',
    'Arkansas-Pine Bluff Golden':   'Arkansas-Pine Bluff',
    'Arkansas-Pine Bluff Golden Lions': 'Arkansas-Pine Bluff',
    'Tennessee State Tigers':       'Tennessee State',
    'Illinois':                     'Illinois',
    'Illinois Fighting Illini':     'Illinois',
    'Bowling Green Falcons':        'Bowling Green',
    'Pennsylvania':                 'Pennsylvania',
    'North Carolina State Wolfpack': 'North Carolina State',
    'NC State Wolfpack':            'North Carolina State',
    'Western Kentucky Hilltoppers': 'Western Kentucky',
    'Western Kentucky':             'Western Kentucky',
    'Tulane Green Wave':            'Tulane',
    'Tulane':                       'Tulane',
    'Youngstown State Penguins':    'Youngstown State',
    'Youngstown State':             'Youngstown State',
    'Bucknell':                     'Bucknell',
    'Nebraska Cornhuskers':         'Nebraska',
    'Stanford Cardinals':           'Stanford',
    'Stanford Cardinal':            'Stanford',
    'Stanford':                     'Stanford',
    'Colorado State Rams':          'Colorado State',
    'Kansas Jayhawks':              'Kansas',
    'West Florida Argonauts':       'West Florida',
}
 
# ── Set of known colleges (standardized names) ────────────────────────────────
# Anything NOT in this set after mapping will be flagged as non-college.
KNOWN_COLLEGES = set(SCHOOL_MAP.values())
 
# ── Apply standardization ─────────────────────────────────────────────────────
def standardize_school(raw_name):
    """Return (standardized_name, is_college) for a raw school string."""
    cleaned = raw_name.strip()
    standardized = SCHOOL_MAP.get(cleaned, cleaned)   # keep original if no mapping found
    is_college = standardized in KNOWN_COLLEGES
    return standardized, is_college

### 2019 Data Cleaning
---

In [5]:
# Standardize school names and create Is_College flag for 2019 dataset
recruits_2019[['committed_school', 'Is_College']] = recruits_2019['committed_school'].apply(
     lambda x: pd.Series(standardize_school(x)))

In [6]:
recruits_2019[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Spencer Rattler,Oklahoma,True
1,Jayden Daniels,Arizona State,True
2,Bo Nix,Auburn,True
3,Sam Howell,North Carolina,True
4,Max Duggan,TCU,True
5,Ryan Hilinski,South Carolina,True
6,Graham Mertz,Wisconsin,True
7,Taulia Tagovailoa,Alabama,True
8,Dylan Morris,Washington,True
9,Cale Millen,Oregon,True


Looks like for whatever reason, there are issues with D'wan Mathis and Kedon Slovis. 

Both of these guys were committed to schools but te scrapper incorrectly picked them up as null.

Let's fix this.

D'wan Mathis -> Georgia

Kedon Slovis -> USC

In [7]:
# Replace row 43 with Georgia
recruits_2019.loc[43, 'committed_school'] = 'Georgia'
recruits_2019.loc[43, 'Is_College'] = True

# Replace row 68 with USC
recruits_2019.loc[68, 'committed_school'] = 'USC'
recruits_2019.loc[68, 'Is_College'] = True

In [8]:
# Check results
recruits_2019[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Spencer Rattler,Oklahoma,True
1,Jayden Daniels,Arizona State,True
2,Bo Nix,Auburn,True
3,Sam Howell,North Carolina,True
4,Max Duggan,TCU,True
5,Ryan Hilinski,South Carolina,True
6,Graham Mertz,Wisconsin,True
7,Taulia Tagovailoa,Alabama,True
8,Dylan Morris,Washington,True
9,Cale Millen,Oregon,True


## 2020 dataset
---

Let's rinse and repeat this process for the next dataset

In [9]:
# Standardize school names and create Is_College flag for 2020 dataset
recruits_2020[['committed_school', 'Is_College']] = recruits_2020['committed_school'].apply(
     lambda x: pd.Series(standardize_school(x)))

In [10]:
# Look at what players got committed school issues
recruits_2020[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Bryce Young,Mater Dei,False
1,DJ Uiagalelei,Clemson,True
2,C.J. Stroud,Ohio State,True
3,Evan Prater,Cincinnati,True
4,Ja'Quinden Jackson,Texas,True
5,Hudson Card,Texas,True
6,Malik Hornsby,Arkansas,True
7,Harrison Bailey,Tennessee,True
8,Luke Doty,South Carolina,True
9,Haynes King,Texas A&M,True


Players that need fixing:

Bryce Young -> Alabama

Jalen Suggs -> This is a special case as Jalen Suggs chose Basketball over football. He will be dropped from the dataset

TJ Finley -> LSU



In [11]:
# Bryce Young
recruits_2020.loc[0, 'committed_school'] = 'Alabama'
recruits_2020.loc[0, 'Is_College'] = True

# Remove Jalen Suggs
recruits_2020 = recruits_2020.drop(index=31)

# TJ Finley
recruits_2020.loc[38, 'committed_school'] = 'LSU'
recruits_2020.loc[38, 'Is_College'] = True

In [12]:
# Check results
recruits_2020[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Bryce Young,Alabama,True
1,DJ Uiagalelei,Clemson,True
2,C.J. Stroud,Ohio State,True
3,Evan Prater,Cincinnati,True
4,Ja'Quinden Jackson,Texas,True
5,Hudson Card,Texas,True
6,Malik Hornsby,Arkansas,True
7,Harrison Bailey,Tennessee,True
8,Luke Doty,South Carolina,True
9,Haynes King,Texas A&M,True


## 2021 dataset
---

In [13]:
# Standardize school names and create Is_College flag for 2020 dataset
recruits_2021[['committed_school', 'Is_College']] = recruits_2021['committed_school'].apply(
     lambda x: pd.Series(standardize_school(x)))

In [14]:
# Look at what players got committed school issues
recruits_2021[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Quinn Ewers,Ohio State,True
1,Caleb Williams,Oklahoma,True
2,Sam Huard,Washington,True
3,Drake Maye,North Carolina,True
4,Brock Vandagriff,Georgia,True
5,J.J. McCarthy,Michigan,True
6,Kaidon Salter,Tennessee,True
7,Kyle McCord,Ohio State,True
8,Ty Thompson,Oregon,True
9,Tyler Buchner,Notre Dame,True


Players that need adjusting:

Sheduer Sanders -> Jackson State


In [15]:
# Sheduer Sanders
recruits_2021.loc[36, 'committed_school'] = 'Jackson State'
recruits_2021.loc[36, 'Is_College'] = True

In [16]:
# Check results
recruits_2021[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Quinn Ewers,Ohio State,True
1,Caleb Williams,Oklahoma,True
2,Sam Huard,Washington,True
3,Drake Maye,North Carolina,True
4,Brock Vandagriff,Georgia,True
5,J.J. McCarthy,Michigan,True
6,Kaidon Salter,Tennessee,True
7,Kyle McCord,Ohio State,True
8,Ty Thompson,Oregon,True
9,Tyler Buchner,Notre Dame,True


## 2022 recruits
---

In [17]:
# Standardize school names and create Is_College flag for 2022 dataset
recruits_2022[['committed_school', 'Is_College']] = recruits_2022['committed_school'].apply(
     lambda x: pd.Series(standardize_school(x)))

In [18]:
# Look at what players got committed school issues
recruits_2022[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Drew Allar,Penn State,True
1,Cade Klubnik,Clemson,True
2,Conner Weigman,Texas A&M,True
3,Ty Simpson,Alabama,True
4,Devin Brown,Ohio State,True
5,Walker Howard,LSU,True
6,Gunner Stockton,Georgia,True
8,Maalik Murphy,Texas,True
9,Brady Allen,Purdue,True
11,Tayven Jackson,Tennessee,True


no issues!

## 2023 recruits
---

In [19]:
# Standardize school names and create Is_College flag for 2022 dataset
recruits_2023[['committed_school', 'Is_College']] = recruits_2023['committed_school'].apply(
     lambda x: pd.Series(standardize_school(x)))

In [20]:
# Look at what players got committed school issues
recruits_2023[['name','committed_school', 'Is_College']]

,name,committed_school,Is_College
0,Arch Manning,Texas,True
1,Nico Iamaleava,Tennessee,True
2,Dante Moore,UCLA,True
3,Jackson Arnold,Oklahoma,True
4,Malachi Nelson,USC,True
5,Jaden Rashada,Arizona State,True
6,Aidan Chiles,Oregon State,True
8,Avery Johnson,Kansas State,True
9,Christopher Vizzina,Clemson,True
10,Lincoln Kienholz,Ohio State,True


## Combine Datasets 
---

Now that the individual datasets are cleaned, lets combine them.

In [21]:
# combine all the datsets into one big dataframe
recruits_all = pd.concat([recruits_2019, recruits_2020, recruits_2021, recruits_2022, recruits_2023], ignore_index=True)

In [22]:
# Check results
recruits_all

,name,url,rank,position,height,weight,composite_rating,school_info,city,state,...,skill_leadership,skill_scorer/finisher,skill_passing/vision,skill_defender,skill_passing,skill_penetration_ability,skill_athleticism,skill_handle,skill_shooter,skill_rebounding
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,PRO,6-1,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,DUAL,6-3,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,DUAL,6-1.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,DUAL,6-0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,DUAL,6-2,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,QB,6-6.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,QB,6-4,190,88,"Milton (Milton, FL)",Milton,FL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,QB,6-2,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,QB,6-2,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Look at null values
recruits_all.isnull().sum()

name                           0
url                            0
rank                           0
position                       0
height                         0
weight                         0
composite_rating               0
school_info                    0
city                           0
state                          0
scouting_report                0
draft_projection               9
reminds_of                    36
evaluated_date                 0
analyst                        1
athletic_background            0
committed_school               0
numerical_rating               0
stars                          0
skill_delivery               123
skill_pocket_presence        122
skill_accuracy               122
skill_intangibles            122
skill_outside_the_pocket     133
skill_mobility               133
skill_arm_strength           122
skill_size                   122
skill_playmaking_ability     140
skill_speed                  140
Is_College                     0
skill_lead

In [24]:
recruits_all.columns

Index(['name', 'url', 'rank', 'position', 'height', 'weight',
       'composite_rating', 'school_info', 'city', 'state', 'scouting_report',
       'draft_projection', 'reminds_of', 'evaluated_date', 'analyst',
       'athletic_background', 'committed_school', 'numerical_rating', 'stars',
       'skill_delivery', 'skill_pocket_presence', 'skill_accuracy',
       'skill_intangibles', 'skill_outside_the_pocket', 'skill_mobility',
       'skill_arm_strength', 'skill_size', 'skill_playmaking_ability',
       'skill_speed', 'Is_College', 'skill_leadership',
       'skill_scorer/finisher', 'skill_passing/vision', 'skill_defender',
       'skill_passing', 'skill_penetration_ability', 'skill_athleticism',
       'skill_handle', 'skill_shooter', 'skill_rebounding'],
      dtype='str')

Looks like the skill ranking columns contain a large % of nulls in the dataset, rendering them unuseable since the number of rows is small to begin with. Therefore, let's drop these columns

In [25]:
# remove columns we don't need
recruits_all = recruits_all.drop(columns=['skill_delivery', 'skill_pocket_presence', 'skill_accuracy',
       'skill_intangibles', 'skill_outside_the_pocket', 'skill_mobility',
       'skill_arm_strength', 'skill_size', 'skill_playmaking_ability',
       'skill_speed', 'Is_College', 'skill_leadership',
       'skill_scorer/finisher', 'skill_passing/vision', 'skill_defender',
       'skill_passing', 'skill_penetration_ability', 'skill_athleticism',
       'skill_handle', 'skill_shooter', 'skill_rebounding'])

In [26]:
# Check results
recruits_all.isnull().sum()

name                    0
url                     0
rank                    0
position                0
height                  0
weight                  0
composite_rating        0
school_info             0
city                    0
state                   0
scouting_report         0
draft_projection        9
reminds_of             36
evaluated_date          0
analyst                 1
athletic_background     0
committed_school        0
numerical_rating        0
stars                   0
dtype: int64

Few remaining nulls in the dataset but could be useful later so they will be kept for now. 

# Feature Engineering
---

FBS or not column:

In [27]:
# Make binary column that measures whether the player is in a fbs program or not

# List of FBS teams
fbs_teams = [
    "Air Force",
    "Akron",
    "Alabama",
    "Appalachian State",
    "Arizona",
    "Arizona State",
    "Arkansas",
    "Arkansas State",
    "Army",
    "Auburn",
    "Ball State",
    "Baylor",
    "Boise State",
    "Boston College",
    "Bowling Green",
    "Buffalo",
    "BYU",
    "California",
    "Central Michigan",
    "Charlotte",
    "Cincinnati",
    "Clemson",
    "Coastal Carolina",
    "Colorado",
    "Colorado State",
    "Connecticut",
    "Duke",
    "East Carolina",
    "Eastern Michigan",
    "FIU",
    "Florida",
    "Florida Atlantic",
    "Florida State",
    "Fresno State",
    "Georgia",
    "Georgia Southern",
    "Georgia State",
    "Georgia Tech",
    "Hawaii",
    "Houston",
    "Illinois",
    "Indiana",
    "Iowa",
    "Iowa State",
    "Jacksonville State",
    "James Madison",
    "Kansas",
    "Kansas State",
    "Kennesaw State",
    "Kent State",
    "Kentucky",
    "Liberty",
    "Louisiana",
    "Louisiana Monroe",
    "Louisiana Tech",
    "Louisville",
    "LSU",
    "Marshall",
    "Maryland",
    "Memphis",
    "Miami",
    "Miami (OH)",
    "Michigan",
    "Michigan State",
    "Middle Tennessee",
    "Minnesota",
    "Mississippi State",
    "Missouri",
    "Missouri State",
    "Navy",
    "NC State",
    "Nebraska",
    "Nevada",
    "New Mexico",
    "New Mexico State",
    "North Carolina",
    "North Carolina State",
    "North Texas",
    "Northern Illinois",
    "Northwestern",
    "Notre Dame",
    "Ohio",
    "Ohio State",
    "Oklahoma",
    "Oklahoma State",
    "Ole Miss",
    "Old Dominion",
    "Oregon",
    "Oregon State",
    "Penn State",
    "Pittsburgh",
    "Purdue",
    "Rice",
    "Rutgers",
    "Sam Houston",
    "San Diego State",
    "San Jose State",
    "SMU",
    "South Alabama",
    "South Carolina",
    "South Florida",
    "Southern Miss",
    "Stanford",
    "Syracuse",
    "TCU",
    "Temple",
    "Tennessee",
    "Texas",
    "Texas A&M",
    "Texas State",
    "Texas Tech",
    "Toledo",
    "Troy",
    "Tulane",
    "Tulsa",
    "UAB",
    "UCLA",
    "UCF",
    "UNLV",
    "USC",
    "Utah",
    "Utah State",
    "UTEP",
    "UTSA",
    "Vanderbilt",
    "Virginia",
    "Virginia Tech",
    "Wake Forest",
    "Washington",
    "Washington State",
    "West Virginia",
    "Western Kentucky",
    "Western Michigan",
    "Wisconsin",
    "Wyoming",
]

# Make column that measures whether the player is in a fbs program or not
recruits_all['Is_FBS'] = 1*(recruits_all['committed_school'].apply(lambda x: x in fbs_teams))

In [28]:
# Check results
recruits_all[['name','committed_school','Is_FBS']]

,name,committed_school,Is_FBS
0,Spencer Rattler,Oklahoma,1
1,Jayden Daniels,Arizona State,1
2,Bo Nix,Auburn,1
3,Sam Howell,North Carolina,1
4,Max Duggan,TCU,1
...,...,...,...
147,Luke Duncan,UCLA,1
148,Emory Williams,Miami,1
149,Jaxon Smolik,Penn State,1
150,Kasen Weisman,Colorado,1


Make Is SEC, Is Big Ten, Is Big 12 and Is ACC columns.

In [29]:
# List of SEC colleges
sec_teams = [
    "Alabama",
    "Arkansas",
    "Auburn",
    "Florida",
    "Georgia",
    "Kentucky",
    "LSU",
    "Mississippi State",
    "Missouri",
    "Oklahoma",
    "Ole Miss",
    "South Carolina",
    "Tennessee",
    "Texas",
    "Texas A&M",
    "Vanderbilt",
]

# Make column that measures whether the player is in a sec program or not
recruits_all['Is_SEC'] = 1*(recruits_all['committed_school'].apply(lambda x: x in sec_teams))

In [30]:
# List of Big Ten colleges
big_ten_teams = [
    "Illinois",
    "Indiana",
    "Iowa",
    "Maryland",
    "Michigan",
    "Michigan State",
    "Minnesota",
    "Nebraska",
    "Northwestern",
    "Ohio State",
    "Oregon",
    "Penn State",
    "Purdue",
    "Rutgers",
    "UCLA",
    "USC",
    "Washington",
    "Wisconsin",
]

# Make column that measures whether the player is in a big10 program or not
recruits_all['Is_Big_Ten'] = 1*(recruits_all['committed_school'].apply(lambda x: x in big_ten_teams))

In [31]:
# list of Big 12 schools
big_XII_teams = [
    "Arizona",
    "Arizona State",
    "Baylor",
    "BYU",
    "Cincinnati",
    "Colorado",
    "Houston",
    "Iowa State",
    "Kansas",
    "Kansas State",
    "Oklahoma State",
    "TCU",
    "Texas Tech",
    "UCF",
    "Utah",
    "West Virginia",
]

# Make column that measures whether the player is in a Big 12 program or not
recruits_all['Is_Big_XII'] = 1*(recruits_all['committed_school'].apply(lambda x: x in big_XII_teams))

In [32]:
# List of ACC schools
acc_teams = [
    "Boston College",
    "California",
    "Clemson",
    "Duke",
    "Florida State",
    "Georgia Tech",
    "Louisville",
    "Miami",
    "NC State",
    "North Carolina",
    "Pittsburgh",
    "SMU",
    "Stanford",
    "Syracuse",
    "Virginia",
    "Virginia Tech",
    "Wake Forest",
]

# Make column that measures whether the player is in a ACC program or not
recruits_all['Is_ACC'] = 1*(recruits_all['committed_school'].apply(lambda x: x in acc_teams))

In [33]:
recruits_all['position'].value_counts()

position
QB      105
PRO      25
DUAL     22
Name: count, dtype: int64

Make dummy variables for each qb category: 'QB', 'PRO', or 'DUAL'

In [34]:
# Make dummy columns for position
recruits_all = pd.get_dummies(recruits_all, columns=['position'], prefix='pos', dtype=int)

In [35]:
# Check changes
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,numerical_rating,stars,Is_FBS,Is_SEC,Is_Big_Ten,Is_Big_XII,Is_ACC,pos_DUAL,pos_PRO,pos_QB
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,6-1,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,9899.5,5,1,1,0,0,0,0,1,0
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,6-3,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,9699.5,4,1,0,0,1,0,1,0,0
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,6-1.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,9599.5,4,1,1,0,0,0,1,0,0
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,6-0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,9499.5,4,1,0,0,0,1,1,0,0
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,6-2,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,9399.5,4,1,0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,6-6.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,8899.5,3,1,0,1,0,0,0,0,1
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,6-4,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,8799.5,3,1,0,0,0,1,0,0,1
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,6-2,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,8699.5,3,1,0,1,0,0,0,0,1
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,6-2,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,8499.5,3,1,0,0,1,0,0,0,1


Make dummy columns for star rating

In [36]:
# Make dummys for star rating
recruits_all = pd.get_dummies(recruits_all, columns=['stars'], prefix='star', dtype=int)

In [37]:
# Check changes
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,Is_Big_Ten,Is_Big_XII,Is_ACC,pos_DUAL,pos_PRO,pos_QB,star_2,star_3,star_4,star_5
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,6-1,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,0,0,0,0,1,0,0,0,0,1
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,6-3,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,0,1,0,1,0,0,0,0,1,0
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,6-1.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,0,0,0,1,0,0,0,0,1,0
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,6-0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,0,0,1,1,0,0,0,0,1,0
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,6-2,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,0,1,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,6-6.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,1,0,0,0,0,1,0,1,0,0
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,6-4,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,0,0,1,0,0,1,0,1,0,0
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,6-2,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,1,0,0,0,0,1,0,1,0,0
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,6-2,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,0,1,0,0,0,1,0,1,0,0


Make Is top five ranked in class column

In [38]:
# Make column of was ranked top five in the country or not
recruits_all['Is_Top_Five_Ranked'] = 1*(recruits_all['rank'] <= 5)

In [39]:
# Check changes
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,Is_Big_XII,Is_ACC,pos_DUAL,pos_PRO,pos_QB,star_2,star_3,star_4,star_5,Is_Top_Five_Ranked
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,6-1,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,0,0,0,1,0,0,0,0,1,1
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,6-3,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,1,0,1,0,0,0,0,1,0,1
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,6-1.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,0,0,1,0,0,0,0,1,0,1
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,6-0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,0,1,1,0,0,0,0,1,0,1
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,6-2,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,1,0,1,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,6-6.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,0,0,0,0,1,0,1,0,0,0
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,6-4,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,0,1,0,0,1,0,1,0,0,0
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,6-2,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,0,0,0,0,1,0,1,0,0,0
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,6-2,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,1,0,0,0,1,0,1,0,0,0


Make Is from west and Is from east columns

In [40]:
# east states list
east_states = [
    "ME", "NH", "VT", "MA", "RI",
    "CT", "NY", "NJ", "PA", "DE",
    "MD", "VA", "WV", "NC", "SC",
    "GA", "FL", "OH", "IN", "MI",
    "WI", "IL", "KY", "TN", "AL", "MS","DC"
]

# west states list
west_states = [
    "MN", "IA", "MO", "AR", "LA", "TX",
    "OK", "KS", "NE", "SD", "ND",
    "MT", "WY", "CO", "NM", "AZ", "UT",
    "ID", "NV", "OR", "WA", "CA", "AK",
    "HI"
]

# Make functionn for whether the player is from the east or west
def categorize_region(state):
    if state in east_states:
        return 'East'
    elif state in west_states:
        return 'West'
    else:
        return 'Other'
    
# Use function to make region column    
recruits_all['Region'] = recruits_all['state'].apply(categorize_region)

# Use get dummies to make region columns
recruits_all = pd.get_dummies(recruits_all, columns=['Region'], prefix='Is_From', dtype=int)

In [41]:
# Check results
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,pos_DUAL,pos_PRO,pos_QB,star_2,star_3,star_4,star_5,Is_Top_Five_Ranked,Is_From_East,Is_From_West
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,6-1,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,0,1,0,0,0,0,1,1,0,1
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,6-3,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,1,0,0,0,0,1,0,1,0,1
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,6-1.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,1,0,0,0,0,1,0,1,1,0
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,6-0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,1,0,0,0,0,1,0,1,1,0
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,6-2,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,1,0,0,0,0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,6-6.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,0,0,1,0,1,0,0,0,0,1
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,6-4,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,0,0,1,0,1,0,0,0,1,0
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,6-2,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,0,0,1,0,1,0,0,0,0,1
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,6-2,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,0,0,1,0,1,0,0,0,1,0


Fix the height column by transforming it into inches

In [42]:
# Make function to acheive this
def height_to_inches(height_str):
    feet, inches = height_str.split('-')
    return int(feet) * 12 + float(inches)

# Apply the changes via the created function to fix the height column
recruits_all['height'] = recruits_all['height'].apply(height_to_inches)

In [43]:
# Check results
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,pos_DUAL,pos_PRO,pos_QB,star_2,star_3,star_4,star_5,Is_Top_Five_Ranked,Is_From_East,Is_From_West
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,73.0,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,0,1,0,0,0,0,1,1,0,1
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,75.0,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,1,0,0,0,0,1,0,1,0,1
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,73.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,1,0,0,0,0,1,0,1,1,0
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,72.0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,1,0,0,0,0,1,0,1,1,0
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,74.0,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,1,0,0,0,0,1,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,78.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,0,0,1,0,1,0,0,0,0,1
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,76.0,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,0,0,1,0,1,0,0,0,1,0
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,74.0,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,0,0,1,0,1,0,0,0,0,1
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,74.0,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,0,0,1,0,1,0,0,0,1,0


Using the height and weight columns, create a weight to height ratio column and a bmi column

In [44]:
# Make sure weight is numeric
recruits_all['weight'] = pd.to_numeric(recruits_all['weight'], errors='coerce')

In [45]:
# Create wieght to height ratio column
recruits_all['weight_to_height_ratio'] = recruits_all['weight'] / recruits_all['height']

# Create bmi column -- note that 703 is the conversion factor to get from imperial to metric units
recruits_all['bmi'] = recruits_all['weight'] / (recruits_all['height'] ** 2) * 703

In [46]:
# Check results
recruits_all

,name,url,rank,height,weight,composite_rating,school_info,city,state,scouting_report,...,pos_QB,star_2,star_3,star_4,star_5,Is_Top_Five_Ranked,Is_From_East,Is_From_West,weight_to_height_ratio,bmi
0,Spencer Rattler,https://247sports.com/player/spencer-rattler-8...,1,73.0,198,99,"Pinnacle (Phoenix, AZ)",Phoenix,AZ,Slightly built prospect with narrow shoulders....,...,0,0,0,0,1,1,0,1,2.712329,26.120098
1,Jayden Daniels,https://247sports.com/player/jayden-daniels-89...,2,75.0,175,97,"Cajon (San Bernardino, CA)",San Bernardino,CA,Possesses a naturally slender build and frame....,...,0,0,0,1,0,1,0,1,2.333333,21.871111
2,Bo Nix,https://247sports.com/player/bo-nix-77945/,3,73.5,207,96,"Pinson Valley (Pinson, AL)",Pinson,AL,Big enough quarterback with some stockiness th...,...,0,0,0,1,0,1,1,0,2.816327,26.937110
3,Sam Howell,https://247sports.com/player/sam-howell-91707/,4,72.0,225,95,"Sun Valley (Monroe, NC)",Monroe,NC,"Thick, stocky build carrying over 220 pounds o...",...,0,0,0,1,0,1,1,0,3.125000,30.512153
4,Max Duggan,https://247sports.com/player/max-duggan-87561/,5,74.0,190,94,"Lewis Central (Council Bluffs, IA)",Council Bluffs,IA,"Muscled, athletic build with no bad weight. Ha...",...,0,0,0,1,0,1,0,1,2.567568,24.391892
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Luke Duncan,https://247sports.com/player/luke-duncan-46104...,29,78.5,185,89,"Miramonte (Orinda, CA)",Orinda,CA,Duncan has been one of the standouts on the 7v...,...,1,0,1,0,0,0,0,1,2.356688,21.105116
148,Emory Williams,https://247sports.com/player/emory-williams-46...,31,76.0,190,88,"Milton (Milton, FL)",Milton,FL,A developmental quarterback prospect with the ...,...,1,0,1,0,0,0,1,0,2.500000,23.125000
149,Jaxon Smolik,https://247sports.com/player/jaxon-smolik-4609...,36,74.0,200,87,"Dowling Catholic (West Des Moines, IA)",West Des Moines,IA,Somewhat limited high school film going into h...,...,1,0,1,0,0,0,0,1,2.702703,25.675676
150,Kasen Weisman,https://247sports.com/player/kasen-weisman-461...,66,74.0,180,85,"South Paulding (Douglasville, GA)",Douglasville,GA,"A gamer that doesn’t seem to lack much, if any...",...,1,0,1,0,0,0,1,0,2.432432,23.108108


### With feature engineering completed, we will export the dataset and begin a new notebook for scouting report analysis. 

In [47]:
# Export cleaned dataset to csv
recruits_all.to_csv(r'../Data/cleaned_recruits_data.csv', index=False)